# Voteview downloads — DW-NOMINATE and roll calls

Voteview publishes Congressional roll-call data as static CSVs: no auth, no
token, no bot filter, and the member file arrives in under a second. So unlike
CDC's downloads this needs no pacing or fingerprinting.

What it needs is a **size gate**. The four files span four orders of
magnitude, and one of them is 700 MB.

In [1]:
import sys
sys.path.append("../")     # notebooks/<source>/ helpers
sys.path.append("../../")  # repo root

import fetch

In [2]:
for name, (path, mb) in fetch.FILES.items():
    default = "fetched by default" if name in fetch.DEFAULT else ""
    gate = "  <- past the size gate" if mb > fetch.SIZE_GATE_MB else ""
    print(f"  {name:10s} {mb:7.1f} MB   {path:32s} {default}{gate}")


  parties        0.1 MB   parties/HSall_parties.csv        fetched by default
  members        6.2 MB   members/HSall_members.csv        fetched by default
  rollcalls     29.8 MB   rollcalls/HSall_rollcalls.csv    
  votes        701.6 MB   votes/HSall_votes.csv              <- past the size gate


`parties` and `members` are the default — 6.3 MB together, and everything the
ideology series need. `parties` is the useful surprise: it carries
`nominate_dim1_median` per party per congress-chamber **already computed**, so
the polarization measure does not require reducing 51,000 member rows.

`votes` is one row per member per roll call and is never fetched by default.
Nothing in the build reads it, and at 700 MB it is a hundred times the rest
combined.

In [3]:
result = fetch.fetch()          # cached: already-present files are skipped
for name, info in result.items():
    print(f"  {name:10s} {info['status']:11s} {info['bytes']/1e6:6.2f} MB")


  parties    cached        0.06 MB
  members    cached        6.20 MB


## Check the panel is intact

Voteview **republishes these files in place** as a Congress advances, so a
download is a snapshot rather than a fixed artifact. A truncated or reshaped
file still parses, so this runs after every fetch and raises rather than
letting a damaged panel through.

In [4]:
import json
print(json.dumps(fetch.verify_members(), indent=1))

{
 "rows": 51064,
 "congresses": "1-119",
 "gaps": [],
 "chambers": {
  "House": 40935,
  "President": 129,
  "Senate": 10000
 },
 "scored": 50840,
 "unscored": 224,
 "float_party_codes": 1669,
 "two_party_from": 34
}


Reading that output:

- **119 congresses, no gaps** — 1789 to 2027, the longest span of any source
  in the stack.
- **129 President rows.** DW-NOMINATE scores presidents too. They are not
  members of either chamber and have to be excluded from chamber aggregates.
- **224 unscored rows** (0.4%) — members with too few votes to place.
- **`two_party_from: 34`.** Democrats and Republicans do not both exist before
  the 34th Congress; earlier there are Federalists, Democratic-Republicans and
  Whigs. So a D-vs-R polarization series covers 1855 onward, **not** the full
  panel — about 85 congresses, not 119.
- **1,669 float-formatted party codes** — see below.

### The trap

`party_code` is `"200"` in most congresses and `"200.0"` in 115–117. Comparing
the raw string drops three modern congresses from any party filter and returns
*nothing* rather than raising — a polarization series with a silent hole
through 2017–2021. `district_code` splits the same way in 3,341 rows.

`fetch.party_code()` is the one place that gets normalised.

In [5]:
import csv
rows = list(csv.DictReader(open("data/HSall_members.csv")))

for congress in (114, 115, 118):
    sel = [r for r in rows if int(r["congress"]) == congress and r["chamber"] == "House"]
    raw = sum(1 for r in sel if r["party_code"] == "200")                 # the trap
    norm = sum(1 for r in sel if fetch.party_code(r) == fetch.REPUBLICAN)  # the fix
    print(f"  congress {congress}  House Republicans: "
          f"raw string match {raw:4d}   normalised {norm:4d}"
          f"{'   <- silently zero' if raw == 0 else ''}")


  congress 114  House Republicans: raw string match  251   normalised  251
  congress 115  House Republicans: raw string match    0   normalised  250   <- silently zero
  congress 118  House Republicans: raw string match  230   normalised  230


## The precomputed medians have holes — compute from members instead

`parties` carries `nominate_dim1_median` per party per congress-chamber, which
looks like the polarization measure for free. It is not: **160 of its 848 rows
have a blank median**, including major parties in congresses 35, 60, 110 and
119. Using it directly would drop those congresses silently.

Computing the median from the member file instead is both complete and
verifiably identical where Voteview does publish one.

In [6]:
import csv, statistics, collections

members = [r for r in csv.DictReader(open("data/HSall_members.csv"))
           if r["chamber"] in fetch.CHAMBERS and r["nominate_dim1"]]
parties = list(csv.DictReader(open("data/HSall_parties.csv")))

scores = collections.defaultdict(list)
for r in members:
    scores[(int(r["congress"]), r["chamber"], fetch.party_code(r))].append(
        float(r["nominate_dim1"]))

agree = disagree = filled = 0
for r in parties:
    key = (int(r["congress"]), r["chamber"], fetch.party_code(r))
    published = (r["nominate_dim1_median"] or "").strip()
    computed = statistics.median(scores[key]) if scores.get(key) else None
    if computed is None:
        continue
    if not published:
        filled += 1
    elif abs(computed - float(published)) <= 0.0005:
        agree += 1
    else:
        disagree += 1

print(f"  agree with the published median : {agree}")
print(f"  disagree                        : {disagree}")
print(f"  blank there, computable here    : {filled}")

  agree with the published median : 584
  disagree                        : 0
  blank there, computable here    : 143


So the reduction reads `members`, and `parties` is a cross-check rather than a
source. The gap between party medians then reproduces the documented
polarization curve — a peak around 1900, the postwar low, and a climb since
the 1970s to the highest values on record.

In [7]:
def median_gap(congress, chamber):
    d = scores.get((congress, chamber, fetch.DEMOCRAT), [])
    r = scores.get((congress, chamber, fetch.REPUBLICAN), [])
    if len(d) < 20 or len(r) < 20:            # before the two-party frame
        return None
    return statistics.median(r) - statistics.median(d)

print("  congress  year    House   Senate")
for c in (35, 60, 80, 90, 100, 110, 115, 119):
    cells = [median_gap(c, ch) for ch in fetch.CHAMBERS]
    print(f"    {c:4d}   {1789 + (c - 1) * 2}  "
          + "  ".join(f"{v:.3f}" if v is not None else "  -  " for v in cells))


  congress  year    House   Senate
      35   1857  0.663  0.689
      60   1907  0.808  0.817
      80   1947  0.498  0.482
      90   1967  0.584  0.587
     100   1987  0.666  0.615
     110   2007  0.805  0.709
     115   2017  0.889  0.800
     119   2025  0.925  0.915


The 1947 trough at 0.498 and the 2025 peak at 0.925 are the two numbers to
recognise: the postwar Congress and the most polarized on record, a spread of
nearly half a DW-NOMINATE unit.

Next: `voteview_series.py` turns these into stored series, loaded by
`db_import` the same way CDC's are. Nothing new is needed on the server —
`timeseries_source_data` already takes a `source`.